### Intermediate+ Starts Here

### The Email SMTP module

Email relies on the "Simple Mail Transfer Protocol", or SMTP, which delineants the rules by which emails are handled by the respective mail servers (Gmail, Yahoo, etc.) and passed around the internet. Python has an [smtplib module](https://docs.python.org/3/library/smtplib.html) that handles this handshake.

smtp host (or server) for popular email providers:
* Gmail: smtp.gmail.com
* Hotmail: smtp.live.com
* Outlook: outlook.office365.com
* Yahoo: smtp.mail.yahoo.com

For example, in 'username@gmail.com', 'username' is the identity of one's email account, while 'gmail.com' is the identity of the mail server. The specific inbox AT the specific server, if you will.

```
import smtplib

email = "sender@gmail.com" # match the server
passsword = "ottfottfottfotff" # 16 digit app password instead of account password

# generate app password in respective server account security and change port
with smtplib.SMTP("smtp.gmail.com", port=587) as connection # server, port number
    connection.starttls() # transport layer security, encryption prevents intercepting
    connection.login(user=email, password=password)
    connection.sendmail(
        from_addr=email, 
        to_addrs="receiver@gmail.com", 
        msg="Subject:Hello\n\This is the body of the email!" # "Subject:" adds subject
        # \n\n snaps to the email contents
    )

```

### The datetime module

The datetime module gets the current date and time

```
import datetime as dt

now = dt.datetime.now() # YYY-MM-DD HH:MM:SS (floating point seconds)
year = now.year()
month = now.month()
day = now.day()
day_of_week = now.weekday() # zero indexing
hour = now.hour()
second = now.second()

# some logic here

```
storing your own datetime:
```
date_of_birth = dt.datetime(year=1998, month=04, day=23) # the rest have default values of 00:00:00
```




In [ ]:
# Send an email with a random motivational quote on Mondays
import smtplib
import datetime as dt
import random

email = "sender@gmil.com"
password = "iejdpekrudnguuan"

day_of_week = dt.datetime.now().weekday()

if day_of_week == 0:
    with open("quotes.txt") as quote_file:
        all_quotes = quote_file.readlines()
        quote = random.choice(all_quotes)

        with smtplib.SMTP("smtp.gmail.com", port=587) as connection:
            connection.starttls()
            connection.login(user=email, password=password)
            connection.sendmail(
                from_addr=email,
                to_addrs="receiver.gmail.com",
                msg=f"Subject: Monday Motivatio\n\n{quote}"
            )

#### Day 32: Automated Birthday Wisher Project

Using a spreadsheet of all the birthdays, with corresponding emails, send a birthday wish from 1 of 3 templates.

Manually running this code every day is impractical. Instead, one can host the following code (and associated files) in [pythonanywhere](https://www.pythonanywhere.com), starting the bash console and running main.py. App password must be reactivated by accessing the link thrown by the error upon first running, ending ... answer/78752", clicking on DisplayUnlockCaptcha (Public Turing Test), proceeding to verify, then re-running the code to login. Finally, schedule the command to run daily at a particular time in UTC under "Tasks" for full automation. On the free tier, you can only schedule one command.

Alternatively, GitHub Workflows does the same for free using the Actions tab on a repository containing main.py and files. Use repository secrets, and use os.environ.get the secrets, such as MY_EMAIL and MY_PASSWORD (so they are not visible to the public). Fill requirements.txt with any packages the script needs. An app password is required as per smtp workflow. Finally, edit ".github.workflows/scheduled.yml" and change the numbers to set schedule:
```
         ┌─ minute (0-59)
         │  ┌─ hour (0-23) in UTC time
         │  │
    cron: '0 9 * * *'   # runs at 9:00 AM UTC every day
```
Test the workflow setup from Actions tab, and then run Daily Script workflow

In [ ]:
import os
import smtplib
import datetime as dt
import pandas
import random
os.chdir('/Users/mujiechen/Library/CloudStorage/Dropbox/Courses/100 Days of Code/Automated Birthday Wisher')

email = "sender@gmail.com"
password = "oemekendjrng"

now = dt.datetime.now()
today = (now.month(), now.day())

data = pandas.read_csv("birthdays.csv")
birthday_dictionary = {(data_row.month,data_row.day):data_row for (index, data_row) in data.iterrows()}

if today in birthday_dictionary:
    birthday_person = birthday_dictionary[today]["name"]
    
    file_path = f"letter_templats/letter_{random.randint(1,3)}.txt"
    with open(file_path) as template:
        contents = template.read()
        contents = contents.replace("[NAME]",birthday_person) # must save to original variable

    recipient = birthday_dictionary[today]["email"]

    with smtplib.SMTP("smtp.gmail.com", port=343) as connection:
        connection.starttls()
        connection.login(user=email, password=password)
        connection.sendmail(from_addr=email, to_addr=recipient, msg=f"Subject: Happy Birtday!\n\n{content}")
    

### API Endpoints and API Parameters

Application Programming Interface, or API, is a set of commands, functions, protocols, and objects that programmers can use to create software relying on or interacting with an external system, i.e. live data hosted in different websites (Yahoo weather, Coinbase prices, NBA updates etc.)

You can use the available objects offered by an API to pull data from a website, using the address, or "API endpoint" the data is hosted. Simple commands for example, would be to get() for broad stroke information, or take parameters for more specific requests (such as sunset/sunrise times at a specific location)

List of APIs [here](https://apilist.fun)


In [17]:
# Kanye Quote Generator using the kanye.rest API
import os
import requests
from tkinter import *
os.chdir('/Users/mujiechen/Library/CloudStorage/Dropbox/Courses/100 Days of Code/Kanye Quotes')

def get_quote():
    response = requests.get(url="https://api.kanye.rest/")
    response.raise_for_status # catch exceptions
    quote = response.json()["quote"]
    canvas.itemconfig(quote_text, text=f"{quote}")

# ui (provided)
window = Tk()
window.title("Kanye Says...")
window.config(padx=50, pady=50)

canvas = Canvas(width=300, height=414)
background_img = PhotoImage(file="background.png")
canvas.create_image(150, 207, image=background_img)
quote_text = canvas.create_text(150, 207, text="Kanye Quote Goes HERE", width=250, font=("Arial", 30, "bold"), fill="white")
canvas.grid(row=0, column=0)

kanye_img = PhotoImage(file="kanye.png")
kanye_button = Button(image=kanye_img, highlightthickness=0, command=get_quote)
kanye_button.grid(row=1, column=0)

window.mainloop()


#### Day 33: International Space Station Overhead Notifier

Uses the API offered by the [Open Notify](http://open-notify.org/Open-Notify-API/ISS-Location-Now/), which returns the current ISS position in JSON (timestamp, request result, longitude and latitude). [LatLong.net](https://www.latlong.net/Show-Latitude-Longitude.html) converts Earth coordinates to an address. It uses the [sunset-sunrise API](https://sunrise-sunset.org/api) to check that its dark enough to see the ISS, before sending the notification.

An API returns [response codes](https://www.webfx.com/web-development/glossary/http-status-codes/) to indicate whether the requests was successful:
```
1XX: Hold On
2XX: Sucess
3XX: Rejected
4XX: Error with Request
5XX: Server Side Error
```
Rather than catching every possible error in if-else statements, the request module has raise_for_status() that can automatically throw the corresponding error message.

In [ ]:
import requests
import datetime as dt
import smtplib
import time

MY_LAT = 51.507351 # floating point numbers from latlong.net for London
MY_LONG = -0.127758
TOLERANCE = 5

MY_EMAIL = "sender@gmail.com"
MY_PASSWORD = "eifnejsiduebchdo"

def is_iss_overhead():
    response = requests.get(url="http://api.open-notify.org/iss-now.json") # location endpoint
    response.raise_for_status() # sends corresponding error message if any
    data = response.json() # accesses JSON data
    
    # access flatpack JSON using "key"
    longitude = float(data["iss_position"]["longitude"])
    latitude = float(data["iss_position"]["latitude"])
    iss_position = (longitude, latitude) # store as tuple
    
    if abs(MY_LAT - iss_position[1]) <= TOLERANCE and abs(MY_LONG - iss_position[0]) <= TOLERANCE:
        return True
    
# check if its past sunset at location
def is_night():
    parameters = { # construct parameters as a dictionary
        "lat": MY_LAT, # keys match documentation requirements
        "lng": MY_LONG, # also date, callback, timezone identifier
        "formatted": 0 # convert from 12 H to unix (24H) time
    } # req by sunset-sunrise API documentaion
    
    response = requests.get(
        url="https://api.sunrise-sunset.org/json", 
        params=parameters
    ) # use ? to start params in a web browser, and & to separate params,
    # then an JSON viewer extention to see the structure
    response.raise_for_status()
    
    # check if past sunset
    hour_now = dt.datetime.now().hour()
    sunrise = response.json()["results"]["sunrise"]
    sunrise_hour = sunrise.split("T")[1].split(":")[0] # the unformatted output has a T in the string
    # further split H/M/S by colon
    sunset = data.json()["results"]["sunset"]
    sunrise_hour = sunset.split("T")[1].split(":")[0]

    if hour_now >= sunset or hour_now <= sunrise:
        return True
        
while True: # run for 24 Hours, ISS orbits 16 times a day
    time.sleep(60)
    if is_iss_overhead() and is_night():
        
        # send an email
        with smtiplib.SMTP("smtp.gmail.com") as connection:
            connection.starttls()
            connection.login(MY_EMAIL, MY_PASSWORD)
            connection.sendmail(
                from_addr=MY_EMAIL, 
                to_addr=MY_EMAIL, 
                msg="Subject: Look Up!\n\nThe ISS is above you in the sky."
            )


### Day 34: GUI Trivia Quiz App

Using the [Open Trivia DB API](https://opentdb.com/api_config.php) to create an app called Quizlet. Users can click on whether they think a statement is True or False, then move on to the next question. Some symbols used by HTML are replaced by [HTML entities](https://www.w3schools.com/html/html_entities.asp) to prevent conflict
> "<" and ">" to  "&lt" and "&gt" <br>
> "&" to "&amp"<br>
> double quotation marks to "&quot"<br>
> single quotation marks to "&apos"<br>
> cent symbol to "&cent"<br>

The [freeformatter API](https://www.freeformatter.com/html-escape.html) can be used to unescape the HTML replacements

In [4]:
import os
import requests
from tkinter import *
os.chdir('/Users/mujiechen/Library/CloudStorage/Dropbox/Courses/100 Days of Code/Quizzler App')
from question_model import Question
from data import question_data
from quiz_brain import QuizBrain
# edit next_question() definition to unescape current_question.text using html.unescape()
# replace user_answer, capture question as output that QuizInterface() calls
from ui import QuizInterface

THEME_COLOR = "#375362"

# get question bank from database using API call (goes into data.py)
parameters = {
    "amount": 5,
    "type": "boolean"
}

response = requests.get(url="https://opentdb.com/api.php", params=parameters)
response.raise_for_status()
question_data = response.json()["results"] # replaces here but just make visible

# logic from Day 17 quiz app
question_bank = []
for question in question_data:
    question_text = question["question"]
    question_answer = question["correct_answer"]
    new_question = Question(question_text, question_answer)
    question_bank.append(new_question)

quiz = QuizBrain(question_bank)

# ui setup as a class (from ui.py)
class QuizInterface:

    def __init__(self, quiz_brain: QuizBrain):
        self.quiz = quiz_brain
    
        self.window = Tk()
        self.window.title("Quizzler")
        self.window.config(bg=THEME_COLOR,padx=20,pady=20)

        self.score_label = Label(text="Score = 0",fg="white",bg=THEME_COLOR)
        self.score_label.grid(column=1,row=0)

        self.canvas = Canvas(width=300, height=250,bg="white")
        self.question_text = self.canvas.create_text(
    		150, 
    		125,
    		text="Text Here",
    		font=("Arial",20,"italic"),
    		fill=THEME_COLOR,
            width=280 # wraps text
            )
        self.canvas.grid(column=0,row=1,columnspan=2,pady=50)

        true_image = PhotoImage(file="images/true.png")
        self.true_button = Button(image=true_image,highlightthickness=0,command=self.true_pressed)
        self.true_button.grid(column=0,row=2)

        false_image = PhotoImage(file="images/false.png")
        self.false_button = Button(image=false_image,highlightthickness=0,command=self.false_pressed)
        self.false_button.grid(column=1,row=2)
    
        self.get_next_question()

        self.window.mainloop()

    def get_next_question(self):
        self.canvas.config(bg="white")
        if self.quiz.still_has_questions():
            self.score_label.config(text=f"Score = {self.quiz.score}")
            q_text = self.quiz.next_question() # returns escaped question
            self.canvas.itemconfig(self.question_text, text=q_text)
        else:
            self.canvas.itemconfig(self.question_text, text="You've reached the end of the quiz")
            self.true_button.config(state="disabled") # prevents further activation
            self.false_button.config(state="disabled")

    def true_pressed(self): # string
        is_right = self.quiz.check_answer("True")
        self.give_feedback(is_right)

    def false_pressed(self):
        is_right = self.quiz.check_answer("False")
        self.give_feedback(is_right)

    def give_feedback(self, is_right):
        if is_right:
            self.canvas.config(bg="green")
        else:
            self.canvas.config(bg="red")
        self.window.after(2000, self.get_next_question)
            

# creating an object from the imported QuizInterface class
quiz_ui = QuizInterface(quiz) # passes Question, including next_question, to QuizInteraface
# next_question passed from main.py, not actually in ui.py

# while quiz.still_has_questions():
#    quiz.next_question()
# cannot have mainloop running within another while loop, instead add to window

print("You've completed the quiz")
print(f"Your final score was: {quiz.score}/{quiz.question_number}")



You've completed the quiz
Your final score was: 4/5


#### Type Hints

As the script gets longer, a way to ensure that variables take the correct data type is to declare a variable's data type.
```
age: int
name: str
height: float
is_human: bool
```
If later you inadvertently "ninteen" to age, IDEs throw an error to say "Expected type 'int', got 'str' instead".
Similarly, you can specify a data type for arguments in a function, and the data type of the output of the function:
```
def police_check(age: int) -> bool:
    if age > 18:
        can_drive = True
    else:
        can_drive = False
    return "They can drive" # type error, 'bool' expected
```
This prevents the wrong data type to be supplied to a downstream function that takes another data type


### API Keys, Authentication & Environment Variables

Some API providers are not free, or costly server side, so they require authentication before to be accessed. For example, [openweathermap](https://home.openweathermap.org/users/sign_up) itself depends on access to about 4000 weather stations globally, and data scientists to process the data in order to predict the weather. API keys is therefore a means of selling privileged data. They usually offer free tiers to sample their technology, restricting API users using an API key ("appid") which can be later revoked. Different tiers are hosted on different API endpoints.

[Online JSON viewer](https://jsonviewer.stack.hu)

#### Send SMS using Twilio API

The [Twilio API](https://www.twilio.com/docs/messaging/quickstart) is a paid service that allows SMS to be sent from a virtual phone numbers to anyone in the world, offering trial credits to test their service. The onboarding process generates a US number, an account_sid, and an AUTH token (in dashboard)

#### Day 35: Rain Alert SMS App

In [ ]:
import requests
from twilio.rest import Client

MY_LAT = 51.507351
MY_LONG = -0.127758

api_key = "4d67edf662a8bf3c1effa1ea5fef4d25" # use secrets / environment variables

account_sid = "account_sid_here"
auth_token = "auth_token_here"

# get weather forecast
parameters = {
    "lat": MY_LAT,
    "lon": MY_LONG,
    "appid": api_key,
    "cnt": 4, # instead of 39 forecasts
}

# get 5 day weather forecast in 3H intervals
response = requests.get(url="https://api.openweathermap.org/data/2.5/forecast",params=parameters)
response.raise_for_status()
weather_data = response.json() # use json viewer

will_rain = False

for hour_data in weather_data["list"]:
    if hour_data["weather"][0]["id"] < 700:
        will_rain = True

if will_rain:
    client = Client(account_sid, auth_token)
    message = client.messages \
        .create(
            body="It's going to rain today. Remember to bring an umbrella ☔️ ",
            from_="trial_twilio_number", # the trial number
            to="", # any verified number
        )

    print(message.status)


As of 31 Jan, 2024, new changes in law prohibits sending of SMS originating from Toll-free numbers within US or Canada unless the TFN has been verified, which can take 2-4 weeks. Instead of virtual phone numbers, Twilio allows messages from its Whatsapp business number, by linking Twilio with Whatsapp, and using instead in the "from" key:
```
...
from="whatsapp:TWILIO_WHATSAPP_NUMBER",
...
```

Finally, the code can be scheduled to run at 7 AM daily by uploading to pythonanywhere's. Free accounts throw an error when testing in bash, as they are issued proxy servers rather than permanent endpoints (paid accounts). Alternate code for specifying html proxies [here](https://help.pythonanywhere.com/pages/TwilioBehindTheProxy/)

<b>Environment Variables</b> are (1) changed outside of and without touching the script to prevent accidental editing (2) sequester any secret keys for security purposes, so they aren't visible if you should upload your codebase somewhere. To store an environment variable:

> (bash) export OWN_API_KEY=emslfekfnekfnkesflnsfe
```
...
"appid": os.environ.get("OWN_API_KEY")
...
```
The command in the scheduled action should then take:
> (bash) export KEY_1=30jfi32f; KEY_2=aoeirjae; ... ; python main.py

This is equivalent to [Github Secrets](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets) (see above)

### Day 36: Stock News Monitoring Project

Professional Stock traders have access to a Bloomberg terminal (~$24,000/year), which provides access to the current prices for a particular stock and any relevant news pertaining that company. Instead, we are cobbling together our own using the [alphavantage](https://www.alphavantage.co), [News](https://newsapi.org) and [Twilio](https://www.twilio.com/en-us) APIs.

The app pulls the stock prices of large cap companies (Apple, Amazon etc.) prices at previous market close, find the different to market close the day before yesterday. Notify you if there is an extraordinary rise or fall. It also fetches some relevant news to explain the change in valuation, which can help you decide if you should buy or sell that particular stock.

In [ ]:
import requests
import datetime as dt
from twilio.rest import Client

# now = dt.datetime.today()
# today_date = now.date()
# yesterday_date = today_date - dt.timedelta(days=1)
# day_before_yesterday_date = yesterday_date - dt.timedelta(days=1)
# y_date_str = str(yesterday_date)
# dby_date_str = str(day_before_yesterday_date)

STOCK = "TSLA"
COMPANY_NAME = "Tesla Inc"
api_key_av = "7TZQCG7VKK50YZCZ"
api_key_news = "1ab282f338f647bd8fae3c6426585a8d"
twilio_sid = ""
twilio_auth_token = ""

## STEP 1: Use https://www.alphavantage.co
# When STOCK price increase/decreases by 5% between yesterday and the day before yesterday then print("Get News").

stock_params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": STOCK,
    "apikey": api_key_av,
}

response = requests.get(url="https://www.alphavantage.co/query", params=stock_params)
response.raise_for_status()
data = response.json()
time_series = data["Time Series (Daily)"]
category_data = [value for (key, value) in time_series.items()] # iterable instead of using datetime
y_close = float(category_data[0]['4. close'])
dby_close = float(category_data[0]['4. close'])

difference = dby_close-y_close
up_down = None
if difference > 0:
    up_down = "↗️"
else:
    up_down = "↘️"

diff_percent = round(difference/float(dby_close)*100)

if abs((dby_close-y_close)/dby_close)>=0.05:
    ## STEP 2: Use https://newsapi.org
    # Instead of printing ("Get News"), actually get the first 3 news pieces for the COMPANY_NAME. 
    news_params = {
        "apiKey": api_key_news,
        "qInTitle": COMPANY_NAME,
    }
    
    news_response = requests.get(url="https://newsapi.org/v2/everything", params=news_parms)
    news_response.raise_for_status()
    all_articles = news_response["articles"]
    three_articles = articles[:3]
    
    ## STEP 3: Use https://www.twilio.com
    # Send a seperate message with the percentage change and each article's title and description to your phone number. 
    # Optional: Format the SMS message like this: 
    """
    TSLA: 🔺2%
    Headline: Were Hedge Funds Right About Piling Into Tesla Inc. (TSLA)?. 
    Brief: We at Insider Monkey have gone over 821 13F filings that hedge funds and prominent investors are required to file by the SEC The 13F filings show the funds' and investors' portfolio positions as of March 31st, near the height of the coronavirus market crash.
    """
    articles = [f"{STOCK_NAME}: {up_down}{diff_percent}%\nHeadlines: {articles['title']}. \nBrief: {article['description']}" for article in three_articles]

    client = Client(twilio_sid, twilio_auth_token)
    for article in articles:
        message = client.messages \
                    .create(
                        body=article,
                        from_="",
                        to="",
                    )

    print(messages.status)


#### API Post, Put, Delete Requests

Other than getting information, APIs can also process [requests from Python](https://requests.readthedocs.io/en/latest/api/) to modify the information on the server.

```
requests.get()
requests.post() # modify or update resource
requests.put() # create or overwrite operations, idempotent
requests.delete()
```
Post can be used to create an account on the server

### Advanced API Authentication using HTTPS Header

Instead of sending the api_key as a parameter that is added to the url (hence visible), store it in the header specified/expected by the API, and pass it to the post request parameters under "headers" argument.


#### Day 37: Habit Tracking Project

Uses the [Pixela API](https://docs.pixe.la) by Japanese developer to track habits both qualitatively and quantitatively

In [ ]:
import requests
from datetime import dt

USERNAME = "username"
TOKEN = "riajeroijeio23r2rjere"
GRAPHD_ID = "graph1"

# add account to server using /users endpoint
pixela_endpoint = "https://pixe.la/v1/users"

user_parameters = {
    "token": TOKEN,
    "username": USERNAME,
    "agreeTermsOfService": "yes",
    "notMinor": "yes",
}

response = requests.post(url=pixel_endpoint, json=user_parameters)
print(response.text)

# create a new graph on pixela
graph_endpoint = f"{pixela_endpoint}/{USERNAME}"

headers = {
    "X-USER-TOKEN": TOKEN # key is specified by API, token is hidden
}

graph_config = {
    "id": GRAPH_ID,
    "name": "Running Graph",
    "unit": "Km",
    "type": "float",
    "color": "sora",
}

requests.post(url=graph_endpoint, json=graph_config, headers=headers)

# new graph is now accesible by graph_endpoint/graph/graph id.html

# add a pixel of data to the graph using post
graph1_endpoint = f"{pixela_endpoint}/{USERNAME}/graphs/{GRAPH_ID}"

# use strftime() to format for compatibility with API requirements
today = dt.datetime.now() # today's entrys
entry = dt.datetime(year=2025, month=4, day=23) # or a past entry

pixel_data = {
    "date": today.strftime("%Y%m%d"), # YYYYmmdd, Year full, month, day
    "quantity": str(input("How many kilometers did you run today?" )), # in the pre-specified units and data type
    "optionalData": "",
}

requests.post(url=graph1_endpoint, json=pixel_data, headers=headers)
print(response.text)

# site will refresh

# update an erroneous pixel using put()
update_endpoint = f"{pixela_endpoint}/{USERNAME}/graphs/{GRAPH_ID}/entry.strftime('%Y%m%d')"

new_pixel_data = 
    "quantity": "10",
}

requests.put(url=update_endpoint, json=new_pixel_data, headers=headers)

# distance becames replaced by quantity

# delete pixels using delete()
delete_endpoint = f"{pixela_endpoint}/{USERNAME}/graphs/{GRAPH_ID}/entry.strftime('%Y%m%d')"

requests.delete(url=delete_endpoint, headers=headers)
# delete does not take any new data


### Day 38: Workout Tracking using Google Sheets

Track workout type, duration, and calories burnt. Running the code in an online console, and asking for input for each field in natural language, upon which the new entry is classified, and the calories are calculated [Nutrition API](https://app.100daysofpython.dev/). Finally, all the data is added to the respective field in a Google Sheet using Sheety. Link Google with Sheety, create new project linked to the specific google sheet, and enable post, put and delete methods.

In [15]:
import requests
from datetime import datetime

GENDER = "male"
WEIGHT_KG = 63
HEIGHT_CM = 165
AGE = 27

# convert into entry using nutrition API
exercise_endpoint = "https://app.100daysofpython.dev/v1/nutrition/natural/exercise"

api_key = "nix_live_ZIzS3Tbc1J0oknHbRHng3TNOMZTNjcMq"
app_id = "app_615a9267da9d4625a9b8b8b3"

headers = {
    "x-app-id": app_id,
    "x-app-key": api_key,
}

exercise_params = {
    "query": input("Describe your exercise: "),
    "gender": GENDER,
    "weight_kg": WEIGHT_KG,
    "height_cm": HEIGHT_CM,
    "age": AGE,
}

response = requests.post(url=exercise_endpoint, json=exercise_params, headers=headers)
result = response.json()

# save entry into google sheets as new row
sheety_endpoint = "https://api.sheety.co/be6c3bb8fbe2252bd0003688deaf86ae/copyOfMyWorkouts/workouts"

today_date = datetime.now().strftime("%d/%m/%Y")
now_time = datetime.now().strftime("%X")

for exercise in result["exercises"]:
    sheet_inputs = {
        "workout": {
            "date": today_date,
            "time": now_time,
            "exercise": exercise["name"].title(),
            "duration": exercise["duration_min"],
            "calories": exercise["nf_calories"],
        }
    }

sheet_response = requests.post(url=sheety_endpoint, json=sheet_inputs)

# then activate either user-pw or bearer authentication. store them as environment variables
# pass either as auth or headers, respectively, inside request parameters


Describe your exercise:  ran 50 km for 10 hours


### Day 39: Flight Deal Finder Capstone Part I

### Day 40: Flight Club Capstone Part II

~<i> a four-lesson break here to cover web foundations</i> ~

### Web Scraping with Beautiful Soup

#### Day 45:

### Day 46: Creating a Spotify Playlist using the Musical Time Machine

### Day 47: Automated Amazon Price Tracker

### Selenium Webdriver Browser

#### Day 48: Game Playing Bot

### Day 49: Automating your Exercise Routine at the Gym

### Day 50: Auto Tinder Swiping Bot

### Day 51: Internet Speed Twitter Complaint Bot

### Day 52: Instagram Follower Bot

### Day 53: Data Entry Job Automation (Web Scraping Capstone)

### Web Development with Flask

#### Day 54:

### HTML and URL Parsing in Flask

#### Day 55: Higher Lower Game

### Rendering HTML/Static Files and Website Templates

#### Day 56:

### Templating with Jinja in Flask Applications

#### Day 57: